# Pipeline Example: LLM-based Entity Extraction

This notebook demonstrates how to use the `Pipeline` class to extract structured entities from text and optionally load them into Neo4j.

**Two modes:**
- **chain** — single LLM call, fast, for bulk processing
- **agent** — ReAct loop exploring the graph before extracting, for complex cases

**Prerequisites:**
- `.env` configured with `LLM_API_KEY` (and optionally `LLM_BASE_URL`)
- Neo4j running (only needed if `load_to_db=True`)

In [ ]:
import sys
sys.path.insert(0, "..")

from src.llm_etl import (
    Pipeline,
    PipelineResult,
    WasteItemExtract,
    FacilityExtract,
    DisposalRuleExtract,
    SourceExtract,
)

## 1. Chain Mode — Basic Extraction

The simplest case: extract entities from text with a single LLM call.

In [ ]:
pipeline = Pipeline(
    schema=WasteItemExtract,
    prompt="Extract all waste items with their disposal targets from the German text.",
    mode="chain",
)

text = """
Altoel muss bei der Schadstoffannahme abgegeben werden.
Altbatterien koennen beim Schadstoffmobil oder im Fachhandel zurueckgegeben werden.
Kartonagen gehoeren in die Altpapiertonne.
Styropor ohne Anhaftungen kommt in die Verpackungstonne.
"""

result = pipeline.run(text)

print(f"Success: {result.success}")
print(f"Entities extracted: {len(result.entities)}")
print(f"Messages: {result.messages}")
print()

for entity in result.entities:
    print(f"  {entity.name} -> {entity.disposal_targets}")

## 2. Chain Mode with DB Loading

Same extraction, but entities are persisted to Neo4j with provenance tracking.
Every entity gets a `DERIVED_FROM` relationship to the source document.

In [ ]:
pipeline = Pipeline(
    schema=WasteItemExtract,
    prompt="Extract waste items with disposal routes.",
    mode="chain",
    load_to_db=True,
    dry_run=True,  # Set to False for actual DB writes
    source=SourceExtract(
        name="example_guide.txt",
        file_type="txt",
        file_path="data/example_guide.txt",
    ),
)

result = pipeline.run(text)

print(f"Entities: {len(result.entities)}")
if result.load_result:
    lr = result.load_result
    print(f"Created: {lr.entities_created}")
    print(f"Updated: {lr.entities_updated}")
    print(f"Skipped: {lr.entities_skipped}")
    print(f"Dry run: {lr.dry_run}")

## 3. Processing a CSV — Loop Pattern

The typical notebook workflow: you handle the source-specific parsing (CSV rows,
PDF pages, API responses), the Pipeline handles the extraction + loading.

This example shows how you'd process a CSV where each row is a product description.

In [ ]:
import csv
from io import StringIO

# Simulated CSV data
csv_data = """name,description
Farbreste,"Fluessige Farbreste muessen bei der Schadstoffannahme abgegeben werden. Eingetrocknete Farbreste koennen in die Restabfalltonne."
Elektrogeraete,"Kleine Elektrogeraete koennen am Wertstoffhof abgegeben werden. Grosse Geraete werden abgeholt."
Gartenabfall,"Gartenabfall kommt in die Biotonne. Groessere Mengen zum Kompostierwerk."
"""

pipeline = Pipeline(
    schema=WasteItemExtract,
    prompt="Extract the waste item and its disposal routes from this description.",
    mode="chain",
)

reader = csv.DictReader(StringIO(csv_data))
all_entities = []

for row in reader:
    result = pipeline.run(
        text=f"{row['name']}: {row['description']}",
        context=f"Product name: {row['name']}",
    )
    all_entities.extend(result.entities)
    print(f"{row['name']}: {len(result.entities)} entities")

print(f"\nTotal: {len(all_entities)} entities from {reader.line_num - 1} rows")

## 4. Agent Mode

The agent explores the knowledge graph before extracting. It searches for existing
entities, checks facility names, and waste stream types so the extraction can
reference exact names from the database.

This requires Neo4j to be running (the agent queries the graph).

In [ ]:
pipeline = Pipeline(
    schema=WasteItemExtract,
    prompt="Extract waste items. Match disposal_targets to exact facility or waste stream names from the graph.",
    mode="agent",
    max_iterations=5,
)

text = """
Altoel und andere fluessige Schadstoffe werden bei der Schadstoffannahme
am Wertstoffhof entgegengenommen. Leere Oelkanister aus Kunststoff kommen
in die Gelbe Tonne.
"""

result = pipeline.run(text)

print(f"Success: {result.success}")
print(f"Entities: {len(result.entities)}")
print()
for msg in result.messages:
    print(f"  {msg}")
print()
for entity in result.entities:
    print(f"  {entity.name} -> {entity.disposal_targets}")

## 5. Custom Schema

You can define your own Pydantic schema for any entity type. The Pipeline
extracts using your schema and the `GenericEntityLoader` handles DB insertion
automatically — no need to register anything.

In [ ]:
from pydantic import BaseModel, Field


class ChemicalExtract(BaseModel):
    """Chemical substance with hazard information."""
    name: str = Field(description="Chemical name in German or IUPAC")
    cas_number: str = Field(default="", description="CAS registry number")
    hazard_class: str = Field(default="", description="GHS hazard classification")
    disposal_notes: str = Field(default="", description="Special disposal instructions")


pipeline = Pipeline(
    schema=ChemicalExtract,
    prompt="Extract chemical substances with their hazard class and disposal instructions.",
    mode="chain",
)

text = """
Aceton (CAS 67-64-1) ist ein leicht entzuendlicher Stoff der Gefahrenklasse 3.
Reste muessen bei der Sonderabfallannahme entsorgt werden.
Natriumhydroxid (CAS 1310-73-2, GHS05 aetzend) darf nicht in den Abfluss.
"""

result = pipeline.run(text)

for entity in result.entities:
    print(f"{entity.name} (CAS: {entity.cas_number})")
    print(f"  Hazard: {entity.hazard_class}")
    print(f"  Disposal: {entity.disposal_notes}")
    print()

## 6. Extracting Different Entity Types

The Pipeline works with any of the built-in schemas. Here we extract facilities
and disposal rules from the same document.

In [ ]:
text = """
Der Wertstoffhof Ost befindet sich in der Hainerweg 11, 60599 Frankfurt.
Oeffnungszeiten: Mo-Fr 7:00-17:00, Sa 8:00-14:00.
Telefon: 069 212-31818.

Farbreste: Wenn fluessig, dann zur Schadstoffannahme.
Wenn eingetrocknet, dann in die Restabfalltonne.
"""

# Extract facilities
facility_pipeline = Pipeline(
    schema=FacilityExtract,
    prompt="Extract disposal facilities with address and opening hours.",
)

fac_result = facility_pipeline.run(text)
print("--- Facilities ---")
for f in fac_result.entities:
    print(f"  {f.name}: {f.address}, {f.opening_hours}")

# Extract disposal rules
rule_pipeline = Pipeline(
    schema=DisposalRuleExtract,
    prompt="Extract conditional disposal rules (if X then Y).",
)

rule_result = rule_pipeline.run(text)
print("\n--- Disposal Rules ---")
for r in rule_result.entities:
    print(f"  {r.waste_item}: if {r.condition} -> {r.if_true_target}, else -> {r.if_false_target}")

## 7. Static Context

You can pass static context that applies to every `run()` call (e.g., known entity names),
plus per-call context for row-specific hints.

In [ ]:
pipeline = Pipeline(
    schema=WasteItemExtract,
    prompt="Extract waste items. Use exact facility/stream names from context.",
    mode="chain",
    context="Known waste streams: Restabfalltonne, Verpackungstonne, Altpapiertonne, Biotonne, Altglascontainer",
)

# Per-call context adds to static context
result = pipeline.run(
    text="Leere Konservendosen gehoeren in die Gelbe Tonne.",
    context="Note: 'Gelbe Tonne' is officially called 'Verpackungstonne'.",
)

for entity in result.entities:
    print(f"{entity.name} -> {entity.disposal_targets}")

## 8. Error Handling

The Pipeline never raises — it returns `PipelineResult` with `success=False` and error details.

In [ ]:
pipeline = Pipeline(
    schema=WasteItemExtract,
    prompt="Extract waste items.",
)

# Empty text
result = pipeline.run("")
print(f"Empty text -> success={result.success}, entities={len(result.entities)}")
print(f"  Message: {result.messages[0]}")

# Whitespace only
result = pipeline.run("   \n\n  ")
print(f"Whitespace -> success={result.success}, entities={len(result.entities)}")

## Summary

| Pattern | When to use |
|---------|-------------|
| `Pipeline(mode="chain")` | Bulk extraction from clean text, CSV rows, API responses |
| `Pipeline(mode="agent")` | Complex text where graph context improves extraction quality |
| `load_to_db=True` | Persist results to Neo4j with provenance tracking |
| `dry_run=True` | Test extraction + loading without writing to DB |
| Custom schema | Any Pydantic BaseModel works, no registration needed |
| Static `context` | Shared knowledge across all runs (known entity names, etc.) |
| Per-call `context` | Row-specific hints merged with static context |